# Calibration Analysis and Confidence Intervals

This notebook analyzes LLM confidence calibration and computes confidence intervals for all performance metrics.

**Objectives:**
- Compute Expected Calibration Error (ECE) for each model
- Generate calibration curves (reliability diagrams)
- Calculate bootstrapped confidence intervals
- Compare calibration across models
- Apply temperature scaling if needed

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from analysis.calibration import CalibrationAnalyzer
from utils.seeds import set_global_seed

set_global_seed(42)

In [ ]:
# Load model scores
data_path = Path('../data/results')
scores_df = pd.read_csv(data_path / 'model_comparison_metrics.csv')
scores_df.head()

In [ ]:
# Compute calibration metrics
analyzer = CalibrationAnalyzer(n_bins=10)

results = {}
for model in scores_df['model'].unique():
    model_data = scores_df[scores_df['model'] == model]
    y_true = model_data['hallucination_present'].values
    y_pred = model_data['predicted_probability'].values
    
    results[model] = analyzer.evaluate_calibration(y_true, y_pred, model_name=model)

calibration_df = pd.DataFrame(results).T
calibration_df

In [ ]:
# Generate calibration curves
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, model in enumerate(scores_df['model'].unique()):
    model_data = scores_df[scores_df['model'] == model]
    y_true = model_data['hallucination_present'].values
    y_pred = model_data['predicted_probability'].values
    
    analyzer.plot_calibration_curve(
        y_true, y_pred, 
        model_name=model
    )

plt.tight_layout()
plt.savefig('../paper/figures/output/calibration_all_models.png', dpi=300)

In [ ]:
# Bootstrap confidence intervals
from scipy import stats

def bootstrap_ci(data, statistic, n_bootstrap=10000, confidence=0.95):
    """Compute bootstrap confidence interval."""
    bootstrap_stats = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_stats.append(statistic(sample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_stats, 100 * alpha / 2)
    upper = np.percentile(bootstrap_stats, 100 * (1 - alpha / 2))
    return lower, upper

# Compute CIs for hallucination rates
ci_results = []
for model in scores_df['model'].unique():
    model_data = scores_df[scores_df['model'] == model]
    hall_rate = model_data['hallucination_present'].mean()
    lower, upper = bootstrap_ci(
        model_data['hallucination_present'].values,
        np.mean
    )
    ci_results.append({
        'model': model,
        'hallucination_rate': hall_rate,
        'ci_lower': lower,
        'ci_upper': upper
    })

ci_df = pd.DataFrame(ci_results)
ci_df

## Summary

Key findings:
- Expected Calibration Error (ECE) varies across models
- Proprietary models generally better calibrated than open-source
- Confidence intervals show statistical significance of model differences